### Active Learning
Active learning is a machine learning technique where the model actively selects the most informative data points to be labeled by an oracle (e.g., a human annotator). This approach is particularly useful when labeling data is expensive or time-consuming, as it allows the model to learn effectively with fewer labeled examples.
#### Key Concepts in Active Learning
1. **Query Strategy**: The method used by the model to select which data points to label. Common strategies include uncertainty sampling, where the model selects data points it is least confident about, and diversity sampling, where the model selects data points that are diverse from those already labeled.
2. **Oracle**: The entity (often a human) that provides labels for the selected data points.
3. **Pool of Unlabeled Data**: The set of data points that the model can choose from to be labeled.
4. **Labeled Data**: The data points that have already been labeled and are used to train the model.
#### Benefits of Active Learning
- **Efficiency**: Active learning can significantly reduce the number of labeled examples needed to achieve a certain level of performance, making it cost-effective.
- **Improved Performance**: By focusing on the most informative data points, active learning can lead to better model performance compared to random sampling.
- **Adaptability**: Active learning can adapt to changing data distributions, allowing the model to remain effective over time.
#### Challenges in Active Learning
- **Selection Bias**: The model may select data points that are not representative of the overall data distribution, leading to biased learning.
- **Oracle Reliability**: The quality of the labels provided by the oracle can affect the performance of the model. If the oracle makes mistakes, it can lead to poor model performance.
- **Computational Cost**: The process of selecting data points and retraining the model can be computationally expensive, especially for large datasets.
#### Conclusion
Active learning is a powerful technique for improving the efficiency and performance of machine learning models, especially in scenarios where labeled data is scarce or expensive to obtain. By strategically selecting the most informative data points for labeling, active learning can help models learn more effectively and adapt to changing data distributions. However, it also comes with challenges that need to be carefully managed to ensure the success of the learning process.

### We will use a simple Fruit Dataset (20 Samples)

Each fruit will contain three features:
	•	type – type of fruit
	•	size – small, medium, large
	•	color – red, green, yellow, blue

We also include a label (poisonous) following the rule: All red fruits are poisonous, while all other colors are either safe or ambiguously poisonous, to obviously poisonous. The dataset will be used to demonstrate the active learning process.

In [96]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv("fruit_active_learning_dataset.csv")
df

,type,size,color,poisonous
0,apple,small,red,1
1,berry,small,blue,0
2,cherry,small,red,1
3,mango,large,yellow,0
4,apple,medium,green,0
...,...,...,...,...
1015,apple,medium,blue,0
1016,apple,medium,blue,0
1017,mango,medium,blue,0
1018,mango,small,yellow,0


Encode categorical features in the unlabeled set using the OneHot Encoder, which is suitable for nominal categorical variables. This will allow us to convert the categorical features into a format that can be used by machine learning algorithms.

In [97]:
encoded_df = pd.get_dummies(df[["type", "size", "color"]]).replace({True: 1, False: 0})
encoded_df["poisonous"] = df["poisonous"]
print("\nEncoded DataFrame:")
encoded_df


Encoded DataFrame:


/var/folders/gc/49grx5f90nv3f12yh4jhhf740000gn/T/ipykernel_24824/1506310626.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encoded_df = pd.get_dummies(df[["type", "size", "color"]]).replace({True: 1, False: 0})


,type_apple,type_berry,type_cherry,type_mango,size_large,size_medium,size_small,color_blue,color_green,color_red,color_yellow,poisonous
0,1,0,0,0,0,0,1,0,0,1,0,1
1,0,1,0,0,0,0,1,1,0,0,0,0
2,0,0,1,0,0,0,1,0,0,1,0,1
3,0,0,0,1,1,0,0,0,0,0,1,0
4,1,0,0,0,0,1,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1,0,0,0,0,1,0,1,0,0,0,0
1016,1,0,0,0,0,1,0,1,0,0,0,0
1017,0,0,0,1,0,1,0,1,0,0,0,0
1018,0,0,0,1,0,0,1,0,0,0,1,0


Get target and features. This will be used to train the model and make predictions. The target variable is 'poisonous', and the features are the encoded versions of 'type', 'size', and 'color'.

In [98]:
target = 'poisonous'
features = encoded_df.columns.drop(target)
print("\nFeatures:", features)
print("Target:", target)

# Convert to numpy arrays for model training
X_encoded = encoded_df[features].values
y = encoded_df[target].values
print("\nEncoded Features Shape:", X_encoded.shape)
print("Target Shape:", y.shape)


Features: Index(['type_apple', 'type_berry', 'type_cherry', 'type_mango', 'size_large',
       'size_medium', 'size_small', 'color_blue', 'color_green', 'color_red',
       'color_yellow'],
      dtype='object')
Target: poisonous

Encoded Features Shape: (1020, 11)
Target Shape: (1020,)


For Active Learning we will split the dataset into:
1. **Labeled Set**: A small subset of the data that is initially labeled (e.g., 5 samples).
2. **Unlabeled Set**: The remaining data that is not labeled and will be used for active learning (e.g., 15 samples).

In [99]:
from sklearn.model_selection import train_test_split
# Split the dataset into labeled and unlabeled sets
labeled_set, unlabeled_set = train_test_split(X_encoded, test_size=0.75, random_state=42)
# Return the length of the labeled and unlabeled sets
print("\nLabeled Set Length:", len(labeled_set))
print("Unlabeled Set Length:", len(unlabeled_set))
# Using indexing to separate features and target for both sets
X_labeled = labeled_set[:, :-1]  # Features for the labeled set [select all rows, all columns except the last one]
y_labeled = labeled_set[:, -1]   # Target for the labeled set [select all rows, only the last column]
X_unlabeled = unlabeled_set[:, :-1]  # Features for the unlabeled set [select all rows, all columns except the last one]
y_unlabeled = unlabeled_set[:, -1]   # Target for the unlabeled set [select all rows, only the last column] (not used in training, for evaluation)


Labeled Set Length: 255
Unlabeled Set Length: 765


In Active Learning, the labeled set will be used to train an initial model, and the model will then query the unlabeled set to select the most informative samples for labeling. This process will be repeated iteratively until a satisfactory model performance is achieved or a certain number of queries have been made.

Train a Logistic Regression model on the labeled set and evaluate its performance on the unlabeled set. We will then use an active learning strategy to select the most informative samples from the unlabeled set for labeling and retrain the model iteratively.

In [100]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression()

The active learning process will involve the following steps:
1. Train the model on the labeled set.
2. Use the model to make predictions on the unlabeled set and calculate the uncertainty of each prediction.
3. Select the most uncertain samples from the unlabeled set for labeling.
4. Add the newly labeled samples to the labeled set and remove them from the unlabeled set.
5. Repeat the process until a stopping criterion is met (e.g., a certain number of iterations or a performance threshold).

In [101]:
def uncertainty_sampling(model, X_pool, k=5):
    probs = model.predict_proba(X_pool)
    uncertainty = 1 - np.max(probs, axis=1)
    k = min(k, len(X_pool))
    query_idx = np.argsort(uncertainty)[-k:]
    return query_idx


iterations = 10
for i in range(iterations):
    print(f"\nIteration {i+1}")
    # Train model
    model.fit(X_labeled, y_labeled)
    # Select uncertain samples
    query_idx = uncertainty_sampling(model, X_unlabeled, k=5)
    # Add queried samples to labeled set
    X_labeled = np.vstack((X_labeled, X_unlabeled[query_idx]))
    y_labeled = np.concatenate((y_labeled, y_unlabeled[query_idx]))
    # Remove queried samples from pool
    X_unlabeled = np.delete(X_unlabeled, query_idx, axis=0)
    y_unlabeled = np.delete(y_unlabeled, query_idx, axis=0)
    
    print("Added samples:", len(query_idx))
    print("Labeled set size:", len(X_labeled))
    print("Remaining unlabeled:", len(X_unlabeled))

y_pred = model.predict(X_unlabeled)
print("\nClassification Report on Remaining Unlabeled Set:")
print(classification_report(y_unlabeled, y_pred))


Iteration 1
Added samples: 5
Labeled set size: 260
Remaining unlabeled: 760

Iteration 2
Added samples: 5
Labeled set size: 265
Remaining unlabeled: 755

Iteration 3
Added samples: 5
Labeled set size: 270
Remaining unlabeled: 750

Iteration 4
Added samples: 5
Labeled set size: 275
Remaining unlabeled: 745

Iteration 5
Added samples: 5
Labeled set size: 280
Remaining unlabeled: 740

Iteration 6
Added samples: 5
Labeled set size: 285
Remaining unlabeled: 735

Iteration 7
Added samples: 5
Labeled set size: 290
Remaining unlabeled: 730

Iteration 8
Added samples: 5
Labeled set size: 295
Remaining unlabeled: 725

Iteration 9
Added samples: 5
Labeled set size: 300
Remaining unlabeled: 720

Iteration 10
Added samples: 5
Labeled set size: 305
Remaining unlabeled: 715

Classification Report on Remaining Unlabeled Set:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       577
           1       1.00      1.00      1.00       138

    accuracy  